# 📊 Week 1: Exploratory Data Analysis
## Telco Customer Churn — IBM Sample Dataset

**Goal:** Understand the dataset, identify data quality issues, and surface the most important patterns before modelling.

---
### Sections
1. Setup & Data Loading
2. Data Quality Audit
3. Target Variable — Churn Rate & Class Imbalance
4. Numerical Features Analysis
5. Categorical Features Analysis
6. Correlation & Relationships
7. Key Insights Summary

## 1. Setup & Data Loading

In [6]:
import sys
from pathlib import Path

# Add project root to path so src/ imports work
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data.load_data import load_raw, data_summary

# ── Plotting style ─────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0F1117',
    'axes.facecolor':   '#1A1D27',
    'axes.edgecolor':   '#2E3147',
    'axes.labelcolor':  '#C8CDE4',
    'xtick.color':      '#8B91B3',
    'ytick.color':      '#8B91B3',
    'text.color':       '#E0E4F5',
    'grid.color':       '#2E3147',
    'grid.linestyle':   '--',
    'grid.alpha':       0.6,
    'font.family':      'sans-serif',
    'font.size':        11,
})
PALETTE = ['#6C63FF', '#FF6B9D', '#43E97B', '#FFB347', '#4FC3F7']
print('Setup complete ✓')

Setup complete ✓


In [7]:
df = load_raw()
summary = data_summary(df)

print(f"Shape      : {summary['shape']}")
print(f"Churn rate : {summary['churn_rate']:.1%}")
print(f"Missing    : {summary['missing']}")
df.head()

FileNotFoundError: Raw data not found at /Users/leonberger/Documents/comp sci /DataSci/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv.
Run:  kaggle datasets download -d blastchar/telco-customer-churn -p data/raw --unzip

## 2. Data Quality Audit

In [8]:
# ── Missing values ────────────────────────────────────────────────────────
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

print('=== Missing Values ===')
display(missing_df)

print(f'\nDuplicates: {df.duplicated().sum()}')
print(f'Unique customers: {df["customerID"].nunique()}')

NameError: name 'df' is not defined

In [ ]:
# ── Data types ─────────────────────────────────────────────────────────────
print('=== Data Types ===')
print(df.dtypes.to_string())

In [ ]:
# ── Numerical summary ─────────────────────────────────────────────────────
df.describe().T.style.background_gradient(cmap='Blues')

## 3. Target Variable — Churn Rate & Class Imbalance

In [9]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Customer Churn Distribution', fontsize=16, fontweight='bold', y=1.02)

# Bar chart
bars = axes[0].bar(churn_counts.index, churn_counts.values,
                   color=[PALETTE[0], PALETTE[1]], edgecolor='none', width=0.5)
for bar, pct in zip(bars, churn_pct.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
                 f'{pct:.1f}%', ha='center', fontweight='bold', fontsize=12)
axes[0].set_title('Count by Churn Status', pad=10)
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Number of Customers')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[0].grid(axis='y')

# Donut chart
wedges, texts, autotexts = axes[1].pie(
    churn_counts.values,
    labels=churn_counts.index,
    autopct='%1.1f%%',
    colors=[PALETTE[0], PALETTE[1]],
    startangle=90,
    pctdistance=0.75,
    wedgeprops=dict(width=0.55, edgecolor='#0F1117', linewidth=3)
)
for at in autotexts:
    at.set_fontweight('bold')
axes[1].set_title('Churn Proportion', pad=10)

plt.tight_layout()
plt.savefig(ROOT / 'docs' / 'churn_distribution.png', dpi=150, bbox_inches='tight',
            facecolor='#0F1117')
plt.show()

print(f'\n⚠️  Class imbalance ratio: {churn_counts["No"]/churn_counts["Yes"]:.1f}:1 (No:Yes)')
print('→  Will require SMOTE or class_weight balancing in Week 3')

NameError: name 'df' is not defined

## 4. Numerical Features Analysis

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Numerical Features vs Churn', fontsize=15, fontweight='bold')

for i, col in enumerate(num_cols):
    # Histogram by churn
    for label, color in zip(['No', 'Yes'], [PALETTE[0], PALETTE[1]]):
        axes[0, i].hist(df[df['Churn'] == label][col].dropna(),
                        bins=30, alpha=0.7, color=color, label=label, edgecolor='none')
    axes[0, i].set_title(f'{col} Distribution')
    axes[0, i].set_xlabel(col)
    axes[0, i].legend(title='Churn')
    axes[0, i].grid(axis='y')

    # Box plot
    data_no  = df[df['Churn'] == 'No'][col].dropna()
    data_yes = df[df['Churn'] == 'Yes'][col].dropna()
    bp = axes[1, i].boxplot([data_no, data_yes], labels=['No', 'Yes'],
                             patch_artist=True, medianprops=dict(color='white', linewidth=2))
    for patch, color in zip(bp['boxes'], [PALETTE[0], PALETTE[1]]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[1, i].set_title(f'{col} Box Plot')
    axes[1, i].set_xlabel('Churn')
    axes[1, i].grid(axis='y')

plt.tight_layout()
plt.savefig(ROOT / 'docs' / 'numerical_analysis.png', dpi=150, bbox_inches='tight',
            facecolor='#0F1117')
plt.show()

## 5. Categorical Features Analysis

In [ ]:
cat_cols = ['Contract', 'InternetService', 'PaymentMethod', 'SeniorCitizen',
            'Partner', 'Dependents', 'TechSupport', 'StreamingTV']

fig, axes = plt.subplots(4, 2, figsize=(16, 22))
axes = axes.flatten()
fig.suptitle('Categorical Features — Churn Rate by Category', fontsize=15, fontweight='bold')

for i, col in enumerate(cat_cols):
    churn_by_cat = df.groupby(col)['Churn'].apply(
        lambda x: (x == 'Yes').mean() * 100
    ).sort_values(ascending=False)

    bars = axes[i].barh(churn_by_cat.index, churn_by_cat.values,
                         color=PALETTE[1], alpha=0.8, edgecolor='none')
    axes[i].axvline(x=df['Churn'].eq('Yes').mean()*100, color=PALETTE[0],
                    linestyle='--', linewidth=1.5, label='Overall avg')
    for bar, val in zip(bars, churn_by_cat.values):
        axes[i].text(val + 0.5, bar.get_y() + bar.get_height()/2,
                     f'{val:.1f}%', va='center', fontsize=9)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('Churn Rate (%)')
    axes[i].legend(fontsize=8)
    axes[i].grid(axis='x')

plt.tight_layout()
plt.savefig(ROOT / 'docs' / 'categorical_churn_rates.png', dpi=150, bbox_inches='tight',
            facecolor='#0F1117')
plt.show()

## 6. Tenure × Contract × Churn — Key Business Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Tenure & Charges vs Churn', fontsize=15, fontweight='bold')

# Tenure by Contract Type (churned vs retained)
for label, color in zip(['No', 'Yes'], [PALETTE[0], PALETTE[1]]):
    subset = df[df['Churn'] == label]
    tenure_by_contract = subset.groupby('Contract')['tenure'].mean().sort_values()
    axes[0].barh([f"{c} ({label})" for c in tenure_by_contract.index],
                 tenure_by_contract.values, color=color, alpha=0.8, label=f'Churn={label}')
axes[0].set_xlabel('Average Tenure (months)')
axes[0].set_title('Average Tenure by Contract & Churn')
axes[0].legend()
axes[0].grid(axis='x')

# Monthly Charges scatter coloured by churn
for label, color in zip(['No', 'Yes'], [PALETTE[0], PALETTE[1]]):
    subset = df[df['Churn'] == label]
    axes[1].scatter(subset['tenure'], subset['MonthlyCharges'],
                    alpha=0.25, s=12, color=color, label=f'Churn={label}')
axes[1].set_xlabel('Tenure (months)')
axes[1].set_ylabel('Monthly Charges ($)')
axes[1].set_title('Tenure vs Monthly Charges (coloured by Churn)')
axes[1].legend(markerscale=3)
axes[1].grid()

plt.tight_layout()
plt.savefig(ROOT / 'docs' / 'tenure_charges_churn.png', dpi=150, bbox_inches='tight',
            facecolor='#0F1117')
plt.show()

## 7. Correlation Heatmap

In [ ]:
from src.data.preprocess import drop_ids, handle_missing, encode_target, encode_categoricals

df_enc = (
    df
    .pipe(drop_ids)
    .pipe(handle_missing)
    .pipe(encode_target)
    .pipe(encode_categoricals)
)

corr = df_enc.corr(numeric_only=True)[['Churn']].sort_values('Churn', ascending=False)

fig, ax = plt.subplots(figsize=(6, 12))
colors = [PALETTE[1] if v > 0 else PALETTE[0] for v in corr['Churn']]
ax.barh(corr.index, corr['Churn'], color=colors, alpha=0.85, edgecolor='none')
ax.axvline(0, color='white', linewidth=0.8)
ax.set_title('Feature Correlation with Churn', fontsize=14, fontweight='bold')
ax.set_xlabel('Pearson Correlation')
ax.grid(axis='x')
plt.tight_layout()
plt.savefig(ROOT / 'docs' / 'correlation_with_churn.png', dpi=150, bbox_inches='tight',
            facecolor='#0F1117')
plt.show()

## 8. 📝 Key Insights Summary

| Finding | Detail | Implication |
|---------|--------|-------------|
| **Class Imbalance** | ~26% churn rate | Use SMOTE or `class_weight='balanced'` in Week 3 |
| **Missing Values** | 11 `TotalCharges` NaNs (new customers, tenure=0) | Impute with 0 |
| **Tenure** | Churned customers have much lower tenure | Strong predictor; engineer "early-stage" flag |
| **Contract Type** | Month-to-month customers churn at ~43% vs ~3% for 2-year | Top categorical driver |
| **Monthly Charges** | Higher charges → higher churn risk | Consider charge-to-value ratio feature |
| **Senior Citizens** | Higher churn rate than non-seniors | Flag for targeted retention |
| **Internet Service** | Fiber optic customers churn more | Possibly price or service quality issue |
| **Tech Support** | No tech support → higher churn | Bundle this with retention offers |

---
**Next Steps (Week 2):** Feature engineering based on these signals.